# Cloud Botany: check the Swift reads

Reads the `nan_log.csv` files written by `scripts/cloud_botany_iso.py` (one per member, in
`SAVE_DIR/EXP_<member>/`) and checks:

1. which (member, time) snapshots are done, have no data (run ended) or failed and must be redone,
2. how often reads from Swift were dropped (NaNs that moved between reads) or failed, per field,
3. the NaNs kept as data and masked out of the sums,
4. that the log agrees with the `.nc` files on disk.

Each row of `nan_log.csv` is one read: `status` is `ok` (no NaN), `retry` (NaNs, read again),
`masked` (same NaNs in two reads, kept and masked), `no_data` (NaN everywhere twice) or `error`
(the read raised). The last read of a field is its outcome.

In [ ]:
import getpass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

user = getpass.getuser()
SAVE_DIR = Path(f"/scratch/{user[0]}/{user}/ISO/Cloud_Botany")  # same default as cloud_botany_iso.py
NTIME = 60                                                    # hourly 3D output of nx1536
CRASHED = {7, 15, 38, 39, 47}
SHORT = {11, 14, 43, 46, 87}                                  # runs that end after 48 h
FIELDS = ["thl", "qt", "ql", "qr", "w"]

## Read the logs

In [ ]:
logs = sorted(SAVE_DIR.glob("EXP_*/nan_log.csv"))
log = pd.concat([pd.read_csv(f) for f in logs], ignore_index=True) if logs else pd.DataFrame(
    columns=["member", "time_index", "time", "field", "attempt", "nnan", "status"])
print(f"{len(logs)} members logged, {len(log)} reads")
log.head()

## Outcome of each snapshot

A snapshot is processed in one pass over the fields; a resubmitted job reads it again, so the outcome
is taken from the last pass. A snapshot is **done** when a `.nc` file exists, **no_data** when a field
had no data, and **failed** when a field ended on `retry` or `error` (no consistent read).

In [ ]:
# number the passes over each snapshot: a new pass starts at the first field, attempt 1
log["pass"] = ((log["field"] == FIELDS[0]) & (log["attempt"] == 1)).groupby(
    [log["member"], log["time_index"]]).cumsum()
last_pass = log[log["pass"] == log.groupby(["member", "time_index"])["pass"].transform("max")]
last_read = last_pass.groupby(["member", "time_index", "field"]).tail(1)

def outcome(g):
    if (g["status"] == "no_data").any():
        return "no_data"
    if g["status"].isin(["retry", "error"]).any():
        return "failed"
    return "read"

snap = last_read.groupby(["member", "time_index"]).apply(outcome).rename("outcome").reset_index()
snap["nc"] = [(SAVE_DIR / f"EXP_{m:03d}" / f"CB_ISO_EXP_{m:03d}_TIME_{t:02d}.nc").exists()
              for m, t in zip(snap["member"], snap["time_index"])]
snap.loc[snap["nc"], "outcome"] = "done"
snap["outcome"].value_counts()

In [ ]:
members = sorted(set(range(1, 104)) - CRASHED)
grid = (snap.pivot(index="member", columns="time_index", values="outcome")
        .reindex(index=members, columns=range(NTIME)).fillna("not_run"))
codes = {"done": 0, "no_data": 1, "failed": 2, "not_run": 3, "read": 2}
colors = ["tab:green", "lightgray", "tab:red", "white"]
fig, ax = plt.subplots(figsize=(10, 12))
ax.pcolormesh(grid.columns, np.arange(len(members)), grid.replace(codes).astype(int).values,
              cmap=plt.matplotlib.colors.ListedColormap(colors), vmin=-0.5, vmax=3.5, edgecolors="w", lw=0.2)
ax.set_yticks(np.arange(len(members))[::5], members[::5])
ax.set(xlabel="time index (h)", ylabel="member", title="green done, gray no data, red failed, white not run")
plt.show()

### To redo

Failed snapshots, and snapshots that should have data but were not run (a run that ends after 48 h has
no data from time index 48 on). Resubmitting the members' array tasks redoes them, since finished
times are skipped.

In [ ]:
expected = grid.copy()
for m in SHORT & set(members):
    expected.loc[m, 48:] = "no_data"
todo = expected[(expected == "failed") | (expected == "not_run")].stack().reset_index()
todo.columns = ["member", "time_index", "outcome"]
print(f"{len(todo)} snapshots to redo, in {todo['member'].nunique()} members")
idx = {m: i for i, m in enumerate(members)}  # array task index in cloud_botany_iso.sbatch
print("sbatch --array=" + ",".join(str(idx[m]) for m in sorted(todo["member"].unique())) +
      " scripts/cloud_botany_iso.sbatch")
todo.groupby("member")["time_index"].apply(list)

### Unexpected no-data

A member that is not one of the short runs should have data at every time; `no_data` there could be
a read where every chunk was dropped twice.

In [ ]:
odd = snap[(snap["outcome"] == "no_data") & ~(snap["member"].isin(SHORT) & (snap["time_index"] >= 48))]
odd

## Swift reads per field

`retry` reads are dropped chunks (or NaNs seen once before being confirmed), `error` reads raised an
exception. The retry fraction is per read.

In [ ]:
stats = log.pivot_table(index="field", columns="status", values="attempt", aggfunc="count",
                        fill_value=0).reindex(FIELDS)
stats["reads"] = stats.sum(axis=1)
for s in ("retry", "error"):
    if s in stats:
        stats[f"{s}_fraction"] = stats[s] / stats["reads"]
stats

In [ ]:
dropped = log[log["status"] == "retry"]
print(f"NaNs in discarded reads: {dropped['nnan'].sum():,} in {len(dropped)} reads")
dropped.groupby("field")["nnan"].describe()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))
log[log["status"].isin(["retry", "error"])].groupby("member").size().reindex(members, fill_value=0).plot(
    ax=axs[0], drawstyle="steps-mid", title="retried or failed reads per member")
log[log["status"].isin(["retry", "error"])].groupby("time_index").size().reindex(range(NTIME), fill_value=0).plot(
    ax=axs[1], drawstyle="steps-mid", title="retried or failed reads per time index")
plt.tight_layout()
plt.show()

## NaNs kept as data

`masked` reads had NaNs at the same points in two reads: these points are left out of the sums
(`masked_points` in the `.nc` file). Large or growing counts would deserve a look at the raw data.

In [ ]:
masked = last_read[last_read["status"] == "masked"]
print(f"{len(masked)} field reads with masked NaNs, {masked['nnan'].sum():,} NaNs in total")
masked.pivot_table(index=["member", "time_index"], columns="field", values="nnan", fill_value=0)

## Cross-check with the output files

`NAN_COUNT`, `READS` and `NAN_DISCARDED` in each `.nc` file should match the last pass of the log.

In [ ]:
import xarray as xr

rows = []
for f in sorted(SAVE_DIR.glob("EXP_*/CB_ISO_EXP_*_TIME_*.nc")):
    with xr.open_dataset(f) as ds:
        m, t = int(ds.attrs["member"]), int(ds.attrs["time_index"])
        lp = last_pass[(last_pass["member"] == m) & (last_pass["time_index"] == t)]
        for fld in FIELDS:
            lf = lp[lp["field"] == fld]
            nc = ds.isel(time=0).sel(field=fld)
            rows.append(dict(member=m, time_index=t, field=fld,
                             ok=(int(nc["READS"]) == len(lf)
                                 and int(nc["NAN_COUNT"]) == lf["nnan"].iloc[-1]
                                 and int(nc["NAN_DISCARDED"]) == lf.loc[lf["status"] == "retry", "nnan"].sum())))
check = pd.DataFrame(rows)
print(f"{len(check)} (file, field) pairs, {(~check['ok']).sum() if len(check) else 0} mismatches")
check[~check["ok"]] if len(check) else check